In [1]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats
import seaborn as sns
import pandas as pd
from datetime import datetime
import math
import re
from scipy.stats import norm

from noshow.preprocessing.load_data import (
    load_appointment_csv,
    process_postal_codes,
)
from pathlib import Path
data_path = Path().resolve().parents[0] / "data" / "raw"
appointments_df = load_appointment_csv(data_path / "poliafspraken_no_show.csv")
appointments_df = appointments_df.loc[appointments_df["ziekenhuis"].isin(
        ["HagaZiekenhuis Den Haag"] #, "HagaZiekenhuis Zoetermeer"
    )]

/home/jovyan/No_Show/src/noshow/preprocessing/load_data.py:52: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  appointments_df = pd.read_csv(


In [2]:
# --- Configuration ---
CLINIC_COL = 'hoofdagenda'
confidence = 0.95
Z = norm.ppf(1 - (1 - confidence) / 2)                      # z-score for 95%
TARGET_MOE_PP = 3.0            # desired margin of error in percentage points (±3%)

def n_for_moe_pp(moe_pp, z=Z):
    """
    Calculate minimum sample size n required for a given target margin of error (in % points)
    at 95% confidence for a proportion (worst-case p=0.5).
    """
    P = 0.5
    moe = moe_pp / 100.0  # convert to proportion (e.g., 3% -> 0.03)
    return math.ceil((z**2 * P * (1-P)) / (moe**2))


min_n = n_for_moe_pp(TARGET_MOE_PP)
print(f"Minimum n per clinic for ±{TARGET_MOE_PP:.0f}% precision at 95% confidence ≈ {min_n}")
clinic_counts = appointments_df[CLINIC_COL].value_counts(dropna=False)
print(f"Number of clinics in dataset: {len(clinic_counts)}")
clinic_counts.head()

Minimum n per clinic for ±3% precision at 95% confidence ≈ 1068
Number of clinics in dataset: 45


hoofdagenda
INTERNE GENEESKUNDE    235346
GYNAECOLOGIE           199510
OOGHEELKUNDE           169395
CARDIOLOGIE            118333
CHIRURGIE              105057
Name: count, dtype: int64

In [3]:
#show excluded clinics
clinics_keep = clinic_counts[clinic_counts >= min_n].index
clinics_excluded = clinic_counts[clinic_counts < min_n].index

print(f"Clinics kept: {len(clinics_keep)}")
print(f"Clinics excluded:")
print(clinic_counts[clinic_counts < min_n].sort_values())

Clinics kept: 32
Clinics excluded:
hoofdagenda
VERGAARBAK                   1
RADIOLOGIE DH                1
FYSIOTHERAPIE JKZ            1
MEDISCHE MICROBIOLOGIE       5
REVALIDATIE                 12
DIETETIEK JKZ               17
SEDATIE                     19
MEDISCHE FOTOGRAFIE         90
SEKSUOLOGIE                164
VAATLAB                    555
RADIOLOGIE POLIKLINIEK     839
LOGOPEDIE                  952
KNF                       1037
Name: count, dtype: int64


In [4]:
#Filter based on threshold
appointments_df_filtered = appointments_df[appointments_df[CLINIC_COL].isin(clinics_keep)].copy()

print(f"Rows in original dataset: {len(appointments_df)}")
print(f"Rows after filtering:      {len(appointments_df_filtered)}")
print("Coverage: {:.2f}% of all records retained".format(
    100 * len(appointments_df_filtered) / len(appointments_df)
))

Rows in original dataset: 1718019
Rows after filtering:      1714326
Coverage: 99.79% of all records retained


In [5]:
appointments_df_filtered["no_show"] = "show"

# Setting 'no_show' to 'no_show' for rows with status "Is niet voldaan" and cancelationReason_code "N" or "NF"
appointments_df_filtered.loc[
    (appointments_df_filtered["status"] == "Is niet voldaan") & 
    (appointments_df_filtered["cancelationReason_code"].isin(["N", "NF"])),
    "no_show"
] = "no_show"

# Replacing 'no_show' with 1 and 'show' with 0
appointments_df_filtered["no_show"] = (
    appointments_df_filtered["no_show"].replace({"no_show": "1", "show": "0"}).astype(int)
)

# Grouping by 'hoofdagenda' (clinic) and calculating the no-show count, total count, and unique patients per clinic
no_show_rate = appointments_df_filtered.groupby('hoofdagenda').agg(
    no_show_count=('no_show', 'sum'),
    total_count=('no_show', 'size'),
    unique_patients=('pseudo_id', 'nunique')
).reset_index()

# Calculating the no-show rate for each clinic
no_show_rate['No_Show_Rate'] = no_show_rate['no_show_count'] / no_show_rate['total_count']

# Converting the no-show rate to percentage and formatting it to 3 decimal places
no_show_rate['No_Show_Rate'] = (no_show_rate['No_Show_Rate'] * 100).round(3).astype(str) + '%'

# Adding a column for total appointments per clinic
no_show_rate['Total_Appointments'] = no_show_rate['total_count']

# Displaying the result
display(no_show_rate[['hoofdagenda', 'No_Show_Rate', 'Total_Appointments', 'unique_patients']])


,hoofdagenda,No_Show_Rate,Total_Appointments,unique_patients
0,ANESTHESIOLOGIE,2.994%,5444,4392
1,CARDIOLOGIE,2.706%,118333,38607
2,CARDIOTHORACALE CHIRURGIE,0.412%,8249,3969
3,CHIRURGIE,1.645%,105057,36845
4,DERMATOLOGIE,1.727%,75441,24197
5,DIETETIEK,6.772%,5006,3099
6,FUNCTIE CARDIOLOGIE,44.225%,9160,5931
7,FUNCTIE LONG,35.128%,5107,3281
8,FUNCTIE SCOPIE,13.762%,3517,2513
9,FYSIOTHERAPIE,4.148%,16199,6670


In [6]:
appointments_df_filtered = appointments_df_filtered.loc[appointments_df_filtered["hoofdagenda"].isin(
    ["OOGHEELKUNDE"]
)]

In [7]:
from noshow.features.feature_pipeline import create_features
from noshow.preprocessing.load_data import (
    load_appointment_csv,
    process_appointments,
    process_postal_codes,
)
from noshow.config import CLINIC_CONFIG

In [8]:
all_postalcodes = process_postal_codes("../data/raw/NL.csv")
appointments_df = process_appointments(appointments_df_filtered, CLINIC_CONFIG)
appointments_features = create_features(appointments_df, all_postalcodes)
#168831

before status onbekend 169395
before gender onbekend 169393
after status onbekend 169393
test!!@
before duplicate address_postalcoe onbekend 169393
before start dates NaT onbekend 169393
before start AFTER COVID BIIGGG NaT onbekend 133240
before duplicate appointmnet_id 133240
non LAsT ONE unique index:
133204
START OF ADD PATIENT FEATURESA 133204
AFTER PATIENT FEATURESA 132774
-----
AFTER PHONE FEATURESA 132774
AFTER BIG FILTER 132774
AFTER VNVAST FILTER 131605


In [11]:
test = appointments_features
test["no_show"] = "show"
test.loc[test["cancelationReason_code"].isin(["N", "NF"]), "no_show"] = "no_show"

In [12]:
appointments_df_filtered = test
appointments_df_filtered["no_show"] = "show"

# Setting 'no_show' to 'no_show' for rows with status "Is niet voldaan" and cancelationReason_code "N" or "NF"
appointments_df_filtered.loc[
    (appointments_df_filtered["status"] == "Is niet voldaan") & 
    (appointments_df_filtered["cancelationReason_code"].isin(["N", "NF"])),
    "no_show"
] = "no_show"

# Replacing 'no_show' with 1 and 'show' with 0
appointments_df_filtered["no_show"] = (
    appointments_df_filtered["no_show"].replace({"no_show": "1", "show": "0"}).astype(int)
)

# Grouping by 'hoofdagenda' (clinic) and calculating the no-show count, total count, and unique patients per clinic
no_show_rate = appointments_df_filtered.groupby('hoofdagenda').agg(
    no_show_count=('no_show', 'sum'),
    total_count=('no_show', 'size'),
    unique_patients=('pseudo_id', 'nunique')
).reset_index()

# Calculating the no-show rate for each clinic
no_show_rate['No_Show_Rate'] = no_show_rate['no_show_count'] / no_show_rate['total_count']

# Converting the no-show rate to percentage and formatting it to 3 decimal places
no_show_rate['No_Show_Rate'] = (no_show_rate['No_Show_Rate'] * 100).round(3).astype(str) + '%'

# Adding a column for total appointments per clinic
no_show_rate['Total_Appointments'] = no_show_rate['total_count']

# Displaying the result
display(no_show_rate[['hoofdagenda', 'No_Show_Rate', 'Total_Appointments', 'unique_patients']])


KeyError: "Column(s) ['pseudo_id'] do not exist"

In [9]:
from noshow.features.appointment_features import (
    add_appointments_last_days,
    add_appointments_same_day,
    add_days_since_created,
    add_days_since_last_appointment,
    add_minutes_early,
    add_time_features,
)
from noshow.features.patient_features import add_patient_features

appointments_features = add_appointments_last_days(appointments_features)
appointments_features = add_appointments_same_day(appointments_features)
appointments_features = add_days_since_created(appointments_features)
appointments_features = add_days_since_last_appointment(appointments_features)
appointments_features = add_minutes_early(appointments_features)
appointments_features = add_time_features(appointments_features)
appointments_features = add_patient_features(appointments_features, all_postalcodes)

def select_feature_columns(featuretable: pd.DataFrame) -> pd.DataFrame:
    return featuretable[
        [
            "hour",
            "weekday",
            "minutesDuration",
            "no_show",
            "prev_no_show",
            "prev_no_show_perc",
            "age",
            "dist_umcu",
            "gender",
            "prev_minutes_early",
            "earlier_appointments",
            "appointments_same_day",
            "appointments_last_days",
            "days_since_created",
            "days_since_last_appointment",
        ]
    ]
test["no_show"] = "show"
test.loc[test["cancelationReason_code"].isin(["N", "NF"]), "no_show"] = "no_show"

test["gender"] = (
        test["gender"].replace({"Man": "1", "Vrouw": "0"}).astype(int)
    )
test["status"] = (
    test["status"].replace({"Is voldaan": "1", "Is niet voldaan": "0"}).astype(int)
)

test = select_feature_columns(appointments_features)

print(test.columns)

START OF ADD PATIENT FEATURESA 131605


KeyError: 'latitude'

In [ ]:
# Calculate Pearson correlation between all features and the target variable 'no_show'
correlation_matrix = test.corr()

# Extract the correlation of each feature with 'no_show'
correlation_with_no_show = correlation_matrix['no_show'].sort_values(ascending=False)

# Print the result
print(correlation_with_no_show)

In [ ]:
# List of specific postal codes to check
postal_codes_to_check = ['0000', '1111', '2222', '3333', '4444', '5555', '6666', '7777', '8888', '9999', '1234']

# Filter the DataFrame for rows where the postal code is in the specified list
filtered_postal_codes = appointments_features[appointments_features['address_postalCodeNumbersNL'].isin(postal_codes_to_check)]

# Get the total count for each of the specific postal codes
postal_code_counts = filtered_postal_codes['address_postalCodeNumbersNL'].value_counts()

# Print the total counts for each postal code
print(postal_code_counts)


In [ ]:
import pandas as pd

def calculate_days_difference(row):
    # Convert 'created' to datetime (only date, so no time part)
    created_date = pd.to_datetime(row['created']).date()

    # Convert 'start' to datetime
    start_datetime = pd.to_datetime(row['end'])

    # Calculate the difference in days
    return (start_datetime - pd.Timestamp(created_date)).days

# Apply the function to each row and create a new column 'days_difference'
appointments_features['days_difference'] = appointments_features.apply(calculate_days_difference, axis=1)

# Print the DataFrame with the new 'days_difference' column
print(appointments_features[['created', 'end', 'days_difference']])


In [ ]:
test = appointments_features[appointments_features['afspraak_code'] == 'VNCAT']


test['days_difference'] = test.apply(calculate_days_difference, axis=1)

# Print the DataFrame with the new 'days_difference' column
print(test[['created', 'end', 'days_difference']])
display(test['days_difference'].value_counts())

In [ ]:
print(appointments_features['created'])

In [ ]:


appointments_features["no_show"] = "show"

# Setting 'no_show' to 'no_show' for rows with status "Is niet voldaan" and cancelationReason_code "N" or "NF"
appointments_features.loc[appointments_features["cancelationReason_code"].isin(["N", "NF"]), "no_show"] = "no_show"

# Replacing 'no_show' with 1 and 'show' with 0
appointments_features["no_show"] = (
    appointments_features["no_show"].replace({"no_show": "1", "show": "0"}).astype(int)
)


In [ ]:
print(appointments_features['no_show'].value_counts())

In [ ]:
import matplotlib.pyplot as plt

plt.hist(appointments_features['hour'], bins=24, edgecolor='black')
plt.title('Distribution of Appointments by Hour')
plt.xlabel('Hour of the Day')
plt.ylabel('Number of Appointments')
plt.xticks(range(24))
plt.show()


In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns

# Create a grouped bar plot showing the counts of show vs no_show by hour
sns.countplot(x='hour', hue='no_show', data=appointments_features)
plt.title('No-Show vs Show by Hour of the Day')
plt.xlabel('Hour of the Day')
plt.ylabel('Count')
plt.xticks(range(24))
plt.legend(title='No Show', labels=['Show', 'No Show'])
plt.show()


In [ ]:
hourly_no_show = appointments_features.groupby('hour')['no_show'].mean()

# Plotting the no-show proportion by hour
plt.bar(hourly_no_show.index, hourly_no_show.values)
plt.title('Proportion of No-Shows by Hour of the Day')
plt.xlabel('Hour of the Day')
plt.ylabel('No-Show Proportion')
plt.xticks(range(24))
plt.show()


In [ ]:
# Filter data where hour is between 8 and 16 (inclusive)
filtered_data = appointments_features[(appointments_features['hour'] >= 8) & (appointments_features['hour'] <= 16)]

# Calculate percentage
percentage = (len(filtered_data) / len(appointments_features)) * 100

# Print total values and the percentage
print(f"Total number of values: {len(appointments_features)}")
print(f"Number of values between 8 and 16 hours: {len(filtered_data)}")
print(f"Percentage of data between 8 and 16 hours: {percentage:.2f}%")


In [ ]:
sns.countplot(x='weekday', data=appointments_features)
plt.title('Distribution of Appointments by Weekday')
plt.xlabel('Weekday')
plt.ylabel('Count')
plt.xticks(ticks=range(7), labels=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
plt.show()

In [ ]:
weekday_no_show = appointments_features.groupby('weekday')['no_show'].mean()

# Plotting the no-show proportion by weekday
plt.bar(weekday_no_show.index, weekday_no_show.values, color='salmon')
plt.title('Proportion of No-Shows by Weekday')
plt.xlabel('Weekday')
plt.ylabel('No-Show Proportion')
plt.xticks(ticks=range(7), labels=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
plt.show()

In [ ]:
# Filter data where weekday is not Saturday (5) or Sunday (6)
weekday_data = appointments_features[~appointments_features['weekday'].isin([5, 6])]

# Calculate percentage
percentage = (len(weekday_data) / len(appointments_features)) * 100

# Print total appointments and the percentage
print(f"Total number of appointments: {len(appointments_features)}")
print(f"Number of appointments not on the weekend: {len(weekday_data)}")
print(f"Percentage of appointments not on the weekend: {percentage:.2f}%")


In [ ]:
# Calculate the range of 'days_since_last_appointment'
data_range = appointments_features['days_since_last_appointment'].max() - appointments_features['days_since_last_appointment'].min()

# Define the number of bins
num_bins = 100

# Calculate the bin width
bin_width = data_range / num_bins

print(f"Bin width: {bin_width:.2f} days")


In [ ]:
# Create a histogram of 'days_since_last_appointment'
plt.hist(appointments_features['days_since_last_appointment'], bins=100, edgecolor='black')
plt.title('Distribution of Days Since Last Appointment')
plt.xlabel('Days Since Last Appointment')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Calculate the absolute difference in 'days_since_last_appointment'
appointments_features['prev_appointment'] = appointments_features['days_since_last_appointment'].shift(1)
appointments_features['diff'] = appointments_features['days_since_last_appointment'] - appointments_features['prev_appointment']

# Filter appointments where the difference is <= 18 days
appointments_within_18_days = appointments_features[appointments_features['diff'].abs() <= 18]

# Calculate percentage
percentage = (len(appointments_within_18_days) / len(appointments_features)) * 100

# Print the result
print(f"Percentage of appointments within 18 days of each other: {percentage:.2f}%")


In [ ]:
import seaborn as sns

# Create a box plot for 'days_since_last_appointment'
sns.boxplot(x=appointments_features['days_since_last_appointment'])
plt.title('Box Plot of Days Since Last Appointment')
plt.xlabel('Days Since Last Appointment')
plt.show()


In [ ]:
# Create a violin plot for 'days_since_last_appointment'
sns.violinplot(x=appointments_features['days_since_last_appointment'])
plt.title('Violin Plot of Days Since Last Appointment')
plt.xlabel('Days Since Last Appointment')
plt.show()


In [ ]:
from noshow.visualisation.features_plots import feature_barplot, feature_scatter

In [ ]:
from noshow.features.appointment_features import (
    add_appointments_last_days,
    add_appointments_same_day,
    add_days_since_created,
    add_days_since_last_appointment,
    add_minutes_early,
    add_time_features,
)

In [ ]:
appointments_features = add_minutes_early(appointments_features)

In [ ]:
# Filter data where 'prev_minutes_early' is less than 0 and greater than 0
before_zero = appointments_features[appointments_features['prev_minutes_early'] < 0]
after_zero = appointments_features[appointments_features['prev_minutes_early'] > 0]

# Calculate percentages
before_zero_percentage = (len(before_zero) / len(appointments_features)) * 100
after_zero_percentage = (len(after_zero) / len(appointments_features)) * 100

# Print the results
print(f"Percentage of appointments before 0 minutes: {before_zero_percentage:.2f}%")
print(f"Percentage of appointments after 0 minutes: {after_zero_percentage:.2f}%")


In [ ]:
# Create a histogram of 'prev_minutes_early'
plt.hist(appointments_features['prev_minutes_early'], bins=30, edgecolor='black')
plt.title('Distribution of prev_minutes_early')
plt.xlabel('Minutes Relative to Appointment')
plt.ylabel('Frequency')
plt.show()

In [ ]:
from noshow.features.no_show_features import prev_no_show_features

In [ ]:
appointments_features = prev_no_show_features(appointments_df)

In [ ]:
# Count the occurrences where 'prev_no_show' is greater than 1
count_more_than_1_no_show = len(appointments_features[appointments_features['prev_no_show'] > 1])

# Calculate the percentage
percentage = (count_more_than_1_no_show / len(appointments_features)) * 100

# Print the result
print(f"Percentage of occurrences where 'prev_no_show' > 1: {percentage:.2f}%")


In [ ]:
ax = feature_barplot(
    appointments_features,
    "prev_no_show",
    feature_name="previous no-shows",
)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Create a histogram of 'prev_no_show'
plt.hist(appointments_features['prev_no_show'], bins=30, edgecolor='black')
plt.title('Distribution of prev_no_show')
plt.xlabel('Previous No Shows')
plt.ylabel('Frequency')
plt.show()


In [ ]:
# Create a bar plot for 'prev_no_show'
appointments_features['prev_no_show'].value_counts().sort_index().plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('Distribution of prev_no_show')
plt.xlabel('Previous No Shows')
plt.ylabel('Frequency')
plt.show()


In [ ]:
print(appointments_features.columns)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Create a box plot showing the distribution of 'minutesDuration' for each 'no_show' category
sns.boxplot(x='no_show', y='minutesDuration', data=appointments_features)
plt.title('Minutes Duration vs No Show')dist
plt.xlabel('No Show (0 = Show, 1 = No Show)')
plt.ylabel('Minutes Duration')
plt.show()


In [ ]:
# Create a violin plot showing the distribution of 'minutesDuration' for each 'no_show' category
sns.violinplot(x='no_show', y='minutesDuration', data=appointments_features)
plt.title('Minutes Duration vs No Show')
plt.xlabel('No Show (0 = Show, 1 = No Show)')
plt.ylabel('Minutes Duration')
plt.show()


In [ ]:
# Calculate the mean 'minutesDuration' for each 'no_show' category
mean_duration = appointments_features.groupby('no_show')['minutesDuration'].mean()

# Create a bar plot
mean_duration.plot(kind='bar', color='skyblue')
plt.title('Average Minutes Duration vs No Show')
plt.xlabel('No Show (0 = Show, 1 = No Show)')
plt.ylabel('Average Minutes Duration')
plt.xticks([0, 1], ['Show', 'No Show'], rotation=0)
plt.show()


In [ ]:
# Create a scatter plot of 'minutesDuration' vs 'no_show'
plt.scatter(appointments_features['no_show'], appointments_features['minutesDuration'], alpha=0.5, color='blue')
plt.title('Minutes Duration vs No Show')
plt.xlabel('No Show (0 = Show, 1 = No Show)')
plt.ylabel('Minutes Duration')
plt.show()
